In [24]:
import utils
import numpy as np
import pandas as pd
import os
from scipy.stats import chisquare

import matplotlib.pyplot as plt
import seaborn as sns # Loads in the theme
sns.set_theme()

pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_rows', None)

In [25]:
# Make output folders
output_dir = os.path.join('..', '..','pre-processed', 'data_for_modelling')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [26]:
# Preparation
experiment_output_path = os.path.join('..', '..', '..', 'phd_1_task', 'experiment_output', 'behavioural_files')
#experiment_output_path = os.path.join('..', '..', '..', 'phd_1_task', 'experiment_output_verena')

In [27]:
unique_subject_ids = list(range(819, 820))
unique_sessions = [1]

recent_results = utils.analysis_functions.list_files_with_date_and_subject_id(experiment_output_path, unique_subject_ids, unique_sessions)
print(recent_results)

['../../../phd_1_task/experiment_output/behavioural_files/behavioural_output_sub-819_session-01_dummy_2024-09-18_11-10-42.csv']


In [28]:
# Opens all the files and combines them into one dataframe
combined_results_dataframe = utils.analysis_functions.combine_result_files(recent_results)    
print(combined_results_dataframe)

     trial  emotional_cue_time response objectively_correct  \
0        0          1726650716     down                Even   
1        1          1726650723     down                Even   
2        2          1726650730       up                Even   
3        3          1726650737     down                Even   
4        4          1726650744       up                Even   
5        5          1726650751       up                Even   
6        6          1726650757     down                Even   
7        7          1726650763     down                Even   
8        8          1726650769       up                True   
9        9          1726650776     down                True   
10      10          1726650783       up                True   
11      11          1726650790     down                True   
12      12          1726650797     down                True   
13      13          1726650804       up                True   
14      14          1726650811     down                

# Check congruency, valence and volatility

In [29]:
# Check a block's congruency
combined_results_dataframe['congruency'] = combined_results_dataframe.apply(utils.analysis_functions.check_congruency, args=(True,), axis=1)

# Check a block's valence
combined_results_dataframe['valence'] = combined_results_dataframe.apply(utils.analysis_functions.check_valence, args=(True,), axis=1)

# Check a block's volatility
combined_results_dataframe = combined_results_dataframe[combined_results_dataframe['probability_condition'] != 50]
combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))
combined_results_dataframe = combined_results_dataframe.sort_values(by=['stimuli_type','trial'], ascending=[True, True])
print(combined_results_dataframe[['volatility','probability_condition']])
combined_results_dataframe = combined_results_dataframe.sort_values(by=['subject_id', 'session', 'stimuli_type', 'trial'], ascending=[True, True, True, True])
combined_results_dataframe.reset_index(drop=True, inplace=True)
print(combined_results_dataframe[['subject_id', 'session', 'congruency', 'valence', 'volatility']])
print(combined_results_dataframe.shape[0])

print(combined_results_dataframe['valence'].unique())
print(combined_results_dataframe['congruency'].unique())
print(combined_results_dataframe['volatility'].unique())

    volatility  probability_condition
0            0                     80
1            0                     80
2            0                     80
3            0                     80
4            0                     80
5            0                     80
6            0                     80
7            0                     80
8            0                     80
9            0                     80
10           0                     80
11           0                     80
12           0                     80
13           0                     80
14           0                     80
15           0                     80
16           0                     80
17           0                     80
18           0                     80
19           0                     80
20           0                     80
21           0                     80
22           0                     80
23           0                     80
24           0                     80
25          

/var/folders/bh/ld02p1_550d75y0x32dmpb540000gp/T/ipykernel_2289/877653697.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combined_results_dataframe = combined_results_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))


# Recode existing variables

In [30]:
combined_results_dataframe = combined_results_dataframe.dropna(subset=['response'])

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'up': 2, 'down': 1},
                                                                                    'response',
                                                                                    'response_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {2: 1, 1: 2},
                                                                                    'stimuli_type',
                                                                                    'stimuli_type')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'subjectively_correct',
                                                                                    'subjectively_correct_int')

combined_results_dataframe = utils.analysis_functions.replace_strings_with_integers(combined_results_dataframe,
                                                                                    {'True': 1, 'False': 0},
                                                                                    'objectively_correct',
                                                                                    'objectively_correct_int')

combined_results_dataframe['subject_id'] = combined_results_dataframe['subject_id'].str.replace(r'\D', '', regex=True)

combined_results_dataframe['session'] = combined_results_dataframe['session'].str.replace(r'\D', '', regex=True)

# Now export the data

In [31]:
combined_results_dataframe.to_csv(f'{output_dir}/combined_dataframe_binary_pilot_hard.csv', sep=';', index_label='index')